In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, LongType, ArrayType, StringType

In [0]:
kafka_json_schema = StructType([
    StructField("time", LongType(), True),
    StructField("flight_vector", ArrayType(StringType()), True)
])

In [0]:
kafka_flight_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "13.232.125.130:9092") \
    .option("subscribe", "flight-telemetry") \
    .option("startingOffsets", "earliest") \
    .load()

In [0]:
parsed_flight_df = kafka_flight_df \
    .withColumn("json_string", col("value").cast("string")) \
    .withColumn("parsed_data", from_json(col("json_string"), kafka_json_schema)) \
    .select(
        col("parsed_data.time").alias("time"),
        col("parsed_data.flight_vector").alias("flight_vector"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp")
    )

In [0]:
query = parsed_flight_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "s3://airspace-lakehouse/airspace_pulse/_checkpoints/bronze_flight_states") \
    .toTable("airspace_pulse.bronze.flight_states")

query.awaitTermination()